# Práctica: Preparación y Limpieza de Datos
## Fase 1: Definición del Problema

* **Dataset Seleccionado:** Default of Credit Card Clients (UCI Machine Learning Repository). Contiene información demográfica, historial de pagos y facturación de clientes de tarjetas de crédito en Taiwán.
* **Algoritmo a utilizar:** Regresión Logística (Problema de Clasificación Supervisada).
* **Variable Objetivo:** `default.payment.next.month` (Binaria: 1 = Incurre en impago el próximo mes, 0 = No incurre en impago).
* **Métrica de Evaluación:** Aunque mediremos la *Accuracy* (Exactitud) como métrica base, prestaremos especial atención a métricas como *Recall* o el *F1-Score*. 
    * *Justificación de Negocio:* En detección de fraude o riesgo de crédito, los datasets suelen estar desbalanceados (hay muchos menos morosos que clientes al día). Si el modelo predice que "nadie impaga", la *Accuracy* será alta, pero el modelo será inútil para el banco. Nos interesa minimizar los Falsos Negativos (clientes que impagan pero que el modelo clasificó como buenos).
* **Objetivo de Negocio:** Reducir el riesgo crediticio (NPL - Non-Performing Loans). Al identificar proactivamente qué clientes tienen alta probabilidad de impagar su tarjeta de crédito el próximo mes, el banco (ej. BBVA) puede activar protocolos de contención: reducir límites de crédito, ofrecer refinanciación o lanzar campañas de educación financiera preventiva, protegiendo así el balance del banco.

### Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings

In [2]:
# Ignoramos warnings temporales para mantener el notebook limpio para la presentación
warnings.filterwarnings('ignore') 

### Read the Data
- Describe each column
- rename the columns' names for easy use.

In [ ]:
### Read the Data
# 1. Cargamos los datos omitiendo la primera fila de metadatos (header=1), ya que La primera fila (índice 0) contiene nombres genéricos (X1, X2, Y)
file_path = 'data/default_of_credit_card_clients.xls'
df = pd.read_excel(file_path, header=1)

In [4]:
# 2. Estandarizamos los nombres de las columnas a 'snake_case'
df.columns = [col.lower().replace(' ', '_').replace('.', '_') for col in df.columns]

# 3. Renombramos la variable objetivo a algo más corto y claro para el negocio
df.rename(columns={'default_payment_next_month': 'default_payment'}, inplace=True)

# 4. Revisamos que la carga fue exitosa
print(f"Dimensiones iniciales del dataset: {df.shape}")
display(df.head())

Dimensiones iniciales del dataset: (30000, 25)


,id,limit_bal,sex,education,marriage,age,pay_0,pay_2,pay_3,pay_4,...,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,default_payment
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


### Understanding and Examinate Data

In [6]:
# 1. Revisión de tipos de datos y valores no nulos
print("--- Información general del Dataset ---")
df.info()
print("\n")

# 2. Estadísticas descriptivas (para detectar anomalías a simple vista)
# Transponemos (.T) para que sea más fácil de leer en la pantalla
print("--- Estadísticas Descriptivas ---")
display(df.describe().T)

# 3. ¿Tenemos filas duplicadas exactas?
duplicados = df.duplicated().sum()
print(f"\nNúmero de filas exactamente duplicadas: {duplicados}")

--- Información general del Dataset ---
<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               30000 non-null  int64
 1   limit_bal        30000 non-null  int64
 2   sex              30000 non-null  int64
 3   education        30000 non-null  int64
 4   marriage         30000 non-null  int64
 5   age              30000 non-null  int64
 6   pay_0            30000 non-null  int64
 7   pay_2            30000 non-null  int64
 8   pay_3            30000 non-null  int64
 9   pay_4            30000 non-null  int64
 10  pay_5            30000 non-null  int64
 11  pay_6            30000 non-null  int64
 12  bill_amt1        30000 non-null  int64
 13  bill_amt2        30000 non-null  int64
 14  bill_amt3        30000 non-null  int64
 15  bill_amt4        30000 non-null  int64
 16  bill_amt5        30000 non-null  int64
 17  bill_amt6        3000

,count,mean,std,min,25%,50%,75%,max
id,30000.0,15000.500000,8660.398374,1.0,7500.75,15000.5,22500.25,30000.0
limit_bal,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
sex,30000.0,1.603733,0.489129,1.0,1.00,2.0,2.00,2.0
education,30000.0,1.853133,0.790349,0.0,1.00,2.0,2.00,6.0
marriage,30000.0,1.551867,0.521970,0.0,1.00,2.0,2.00,3.0
age,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
pay_0,30000.0,-0.016700,1.123802,-2.0,-1.00,0.0,0.00,8.0
pay_2,30000.0,-0.133767,1.197186,-2.0,-1.00,0.0,0.00,8.0
pay_3,30000.0,-0.166200,1.196868,-2.0,-1.00,0.0,0.00,8.0
pay_4,30000.0,-0.220667,1.169139,-2.0,-1.00,0.0,0.00,8.0



Número de filas exactamente duplicadas: 0


In [ ]:
# 1. Eliminamos el 'id' ya que no aporta valor predictivo para el negocio
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)
    print("Columna 'id' eliminada con éxito.\n")

# 2. Revisión de tipos de datos y estadísticas base
print("--- Información general del Dataset ---")
df.info()
print("\n--- Estadísticas Descriptivas ---")
display(df.describe().T)

# 3. Identificar y eliminar filas duplicadas
filas_antes = df.shape[0]
df = df.drop_duplicates()
filas_despues = df.shape[0]
print(f"\nFilas duplicadas eliminadas: {filas_antes - filas_despues}")

# 4. Eliminar columnas de valor único o baja varianza
# Utilizamos un umbral conservador (1e-6) sugerido por tu compañero
varianzas = df.select_dtypes(include=[np.number]).var()
columnas_baja_varianza = varianzas[varianzas < 1e-6].index.tolist()
print(f"Columnas eliminadas por baja varianza: {columnas_baja_varianza}")
if columnas_baja_varianza:
    df.drop(columns=columnas_baja_varianza, inplace=True)